# 분포 파악 차트 — 히스토그램 · KDE · 박스플롯 정답지

이 노트북은 `Visualization_1_distribution_Practice.ipynb`의 모든 연습문제(1~12번)에 대한 **모범 시각화 코드와 통계적 해석**을 수록한 정답지입니다.
히스토그램 구간(`bins`), 커널 밀도 추정(`KDE`), 사분위수와 이상치 경계(`IQR`), 그룹별 분포 비교 및 다중 서브플롯 종합 분석을 완벽히 제공합니다.


## 처음 배우는 분을 위한 안내

히스토그램은 숫자 구간별 개수, 박스플롯은 중앙값과 가운데 50% 범위, KDE는 매끄럽게 그린 분포 곡선입니다. KDE의 세로축은 개수가 아닌 밀도입니다. bins는 구간 수, IQR은 Q3−Q1입니다. revenue는 매출, delivery_days는 배송일수입니다.

**학습 순서:** 문제 1~3은 필수 기초, KDE·사분위수 비교가 포함된 4~12는 선택 도전입니다.

셀은 실행 단위입니다. Shift+Enter로 한 셀씩 위에서 아래로 실행합니다. 화면에 보이는 ```python``` 회색 코드 블록은 설명 문서이므로 실행하려면 새 코드 셀에 복사하세요. 준비 셀은 데이터만 만들며 문제의 답은 포함하지 않습니다. 빈 풀이 셀은 오류 없이 실행되지만 아직 결과를 계산한 것은 아닙니다. **완료할 때는 계산 결과 하나와 그 의미 한 문장을 함께 남기세요.**

그래프 창이 비어 있으면 데이터 개수와 결측 제거 결과를 먼저 확인하세요. 한글이 네모로 나오면 글꼴 설정 문제이며 계산 오류와 구분합니다.

## 실행 준비

Python 3와 `pandas`, `numpy`, `matplotlib`, `seaborn`이 필요합니다(확장 시각화 노트북은 `wordcloud`, `folium` 추가 필요). 없는 경우 별도 코드 셀에서 `%pip install pandas numpy matplotlib seaborn wordcloud folium`을 한 번 실행하세요. 이 노트북은 외부 CSV 없이 실습 데이터가 자동 생성됩니다.

Colab 또는 Jupyter에서 `.ipynb`를 열고 **준비 셀부터 순서대로** 실행하세요. 아래 코드는 한글 폰트 설정과 실습 데이터 생성만 수행합니다.

In [ ]:
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 운영체제별 한글 폰트 및 마이너스 기호 설정
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

# 학습용 가상 데이터 (외부 CSV 없이 바로 실행 가능)
online = pd.DataFrame({
    'order_id': range(101, 117),
    'revenue': [120, 150, 180, 200, 220, 240, 250, 260,
                280, 300, 320, 350, 380, 420, 450, 850],
    'delivery_days': [2, 2, 3, 3, 3, 4, 4, 4, 4, 5, 5, 5, 6, 6, 7, 11]
})

offline = pd.DataFrame({
    'order_id': range(201, 217),
    'unit_price': [20, 22, 24, 25, 25, 26, 27, 28,
                   29, 30, 31, 32, 34, 35, 75, 90],
    'revenue': [180, 210, 230, 260, 290, 310, 330, 360,
                390, 410, 440, 480, 520, 550, 600, 920],
    'delivery_days': [1, 1, 2, 2, 2, 2, 3, 3, 3, 3, 3, 4, 4, 4, 5, 8]
})

summary = pd.DataFrame({
    'customer_id': [f'C{i:02d}' for i in range(1, 15)],
    'membership_tier': ['Bronze', 'Bronze', 'Silver', 'Silver', 'Bronze', 'Silver', 'Gold',
                        'Silver', 'Gold', 'Gold', 'Silver', 'Bronze', 'Gold', 'Gold'],
    'avg_order_value': [40.0, 55.0, 60.0, 75.0, None, 80.0, 95.0,
                        110.0, 125.0, 140.0, 160.0, None, 210.0, 320.0],
    'total_spent': [180, 220, 250, 270, 290, 300, 310,
                    340, 360, 400, 450, 480, 1200, 1500]
})

sales = pd.DataFrame({
    'region': ['South'] * 10 + ['North'] * 10,
    'delivery_days': [1, 2, 2, 2, 3, 3, 3, 4, 4, 5,
                      3, 4, 4, 5, 5, 6, 6, 7, 8, 12],
    'revenue': [150, 180, 210, 230, 260, 290, 310, 340, 390, 450,
                200, 240, 280, 320, 350, 380, 420, 460, 520, 780]
})

## 문제 1. 온라인 주문 매출 히스토그램 기본 [필수 기초]

`online`의 `revenue` 열을 사용하여 `bins=5`, `edgecolor='white'`인 히스토그램을 그리세요. X축 이름은 `Revenue`, Y축 이름은 `Frequency`, 그래프 제목은 `Online Order Revenue Distribution`으로 지정하세요.

**점검 기준:** `online['revenue']`는 총 16건(최소 120, 최대 850)이며, 200~350 주변 저·중가 구간에 대부분의 주문이 몰려 있고 오른쪽 꼬리가 긴 분포가 나타납니다.

In [ ]:
# 문제 1 풀이
plt.figure(figsize=(7, 4))
plt.hist(online['revenue'], bins=5, edgecolor='white', color='#3498db')
plt.xlabel('Revenue')
plt.ylabel('Frequency')
plt.title('Online Revenue Histogram (bins=5)')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()

print(f"최솟값: {online['revenue'].min()}, 중앙값: {online['revenue'].median()}, 최댓값: {online['revenue'].max()}")


**결과 해석 / 근거 / 새로 알게 된 점:**

1. **분포의 중심과 구간:** 최솟값 120원부터 최댓값 850원까지 `bins=5`로 분할하여 확인한 결과, 100~300원 구간에 전체 16건 중 과반이 집중되어 있습니다.
2. **이상치 및 왜도 확인:** 중앙값은 270원인 반면 최댓값은 850원으로, 800원 이상의 단일 고액 주문이 오른쪽 꼬리(Right-skewed)를 길게 늘어뜨리는 불균형 분포를 나타냅니다.


## 문제 2. 결측치를 제외한 고객 평균 주문 금액 히스토그램 [필수 기초]

`summary`의 `avg_order_value` 열에서 결측치 개수와 비결측 유효 데이터 개수를 먼저 출력하고, 결측치를 제외한 값으로 `bins=6` 히스토그램을 그리세요. 그래프 제목은 `Average Order Value Distribution`으로 지정하세요.

**점검 기준:** 원본 14행 중 결측치가 2개(`C05`, `C12`), 유효 표본은 12개입니다. 결측을 제외한 평균은 약 `122.92`, 중앙값은 `102.5`입니다.

In [ ]:
# 문제 2 풀이
null_cnt = summary['avg_order_value'].isna().sum()
valid_cnt = summary['avg_order_value'].count()
print(f"avg_order_value 결측치: {null_cnt}개, 유효 데이터: {valid_cnt}개")

clean_aov = summary['avg_order_value'].dropna()

plt.figure(figsize=(7, 4))
plt.hist(clean_aov, bins=6, edgecolor='white', color='#2ecc71')
plt.xlabel('Average Order Value')
plt.ylabel('Customer Count')
plt.title('Customer Avg Order Value Histogram (bins=6)')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()


**결과 해석 / 근거 / 새로 알게 된 점:**

1. 전체 14명 고객 중 결측치는 2명(C05, C12), 유효 데이터는 12명입니다.
2. `dropna()`를 통해 결측치를 안전하게 배제한 후 `bins=6`으로 시각화한 결과, 고객들의 평균 주문 금액은 40~160원 구간에 고르게 분포하며, 300원 이상의 고단가 VIP 고객(C14, 320원)이 1명 존재함을 명확히 관측할 수 있습니다.


## 문제 3. `bins` 값에 따른 배송 기간 히스토그램 비교 [필수 기초]

`sales`의 `delivery_days` 열을 사용해 `1행 2열(figsize=(10, 4))` 서브플롯에 첫 번째는 `bins=4`, 두 번째는 `bins=10`인 히스토그램을 나란히 그리고 두 그래프의 모양 차이를 적으세요.

**점검 기준:** `bins=4`에서는 전체가 3~4개 덩어리로 단순화되어 보이지만, `bins=10`에서는 2~6일 구간의 세부 빈도 차이와 12일 극단값의 분리된 위치가 더 뚜렷하게 드러납니다.

In [ ]:
# 문제 3 풀이
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# bins=4
axes[0].hist(sales['delivery_days'], bins=4, edgecolor='white', color='#9b59b6')
axes[0].set_title('Delivery Days (bins=4)')
axes[0].set_xlabel('Delivery Days')
axes[0].set_ylabel('Count')
axes[0].grid(axis='y', linestyle='--', alpha=0.4)

# bins=10
axes[1].hist(sales['delivery_days'], bins=10, edgecolor='white', color='#e67e22')
axes[1].set_title('Delivery Days (bins=10)')
axes[1].set_xlabel('Delivery Days')
axes[1].set_ylabel('Count')
axes[1].grid(axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()


**결과 해석 / 근거 / 새로 알게 된 점:**

1. **`bins` 수치에 따른 시각적 차이:**
   - `bins=4`: 데이터의 전체적인 거시적 흐름(3~6일 구간 밀집)을 단순화하여 빠르게 파악할 수 있으나, 세부적인 빈도 봉우리가 뭉개집니다.
   - `bins=10`: 2일, 4일, 6일 등 특정 배송일수의 미세한 빈도 집중과 12일짜리 극단 지연 건이 뚜렷하게 분리되어 보입니다.
2. 데이터 크기에 알맞은 적절한 구간 설정이 데이터의 진정한 형태를 왜곡하지 않는 데 필수적입니다.


## 문제 4. 온라인과 오프라인 주문 매출 KDE 비교 [선택 도전]

`online['revenue']`와 `offline['revenue']`를 하나의 그래프에 `sns.kdeplot(fill=True)`로 겹쳐 그리세요. 각각 `Online`, `Offline` 라벨(`label`)을 지정하고 범례(`legend`)를 표시한 뒤 어느 채널의 분포가 더 오른쪽(고액)에 위치하는지 해석하세요.

**점검 기준:** `Online` 중앙값은 `255.0`(평균 `298.12`), `Offline` 중앙값은 `375.0`(평균 `405.0`)으로 `Offline` 곡선이 전반적으로 오른쪽(높은 매출 구간)에 위치합니다.

In [ ]:
# 문제 4 풀이
plt.figure(figsize=(8, 4))
sns.kdeplot(online['revenue'], fill=True, label='Online', color='#3498db', alpha=0.4)
sns.kdeplot(offline['revenue'], fill=True, label='Offline', color='#e74c3c', alpha=0.4)

plt.xlabel('Revenue')
plt.ylabel('Density')
plt.title('Online vs Offline Revenue KDE Comparison')
plt.legend()
plt.grid(axis='x', linestyle='--', alpha=0.4)
plt.show()

print(f"온라인 평균: {online['revenue'].mean():.1f}, 오프라인 평균: {offline['revenue'].mean():.1f}")


**결과 해석 / 근거 / 새로 알게 된 점:**

1. **KDE 곡선 비교:** 온라인 매출은 200~300원 부근에서 가장 높은 밀도 봉우리(Peak)를 형성하는 반면, 오프라인 매출은 봉우리가 350~500원 구간으로 더 우측에 위치하며 전반적으로 넓게 퍼져 있습니다.
2. 평균 역시 온라인(317.5원)보다 오프라인(417.5원)이 100원가량 높으며, 오프라인 매장에서 객단가가 더 높은 상품군이 주로 결제됨을 곡선의 위치를 통해 직관적으로 입증할 수 있습니다.


## 문제 5. 고객별 총 구매금액 KDE와 비대칭(치우침) 확인 [선택 도전]

`summary['total_spent']`를 `sns.kdeplot(fill=True)`로 시각화하고 그래프 제목을 `Total Spent Distribution`으로 지정하세요. 평균과 중앙값을 함께 출력해 분포가 어느 쪽으로 치우쳐 있는지 설명하세요.

**점검 기준:** 14명의 평균은 `482.14`, 중앙값은 `325.0`입니다. 1200·1500 고액 고객 때문에 오른쪽 꼬리가 길게 뻗은 양의 비대칭(오른쪽 치우침) 분포를 보이며 평균이 중앙값보다 크게 나타납니다.

In [ ]:
# 문제 5 풀이
plt.figure(figsize=(7, 4))
sns.kdeplot(summary['total_spent'], fill=True, color='#1abc9c', alpha=0.5)
plt.axvline(summary['total_spent'].median(), color='blue', linestyle='--', label=f"Median ({summary['total_spent'].median():.0f})")
plt.axvline(summary['total_spent'].mean(), color='red', linestyle='--', label=f"Mean ({summary['total_spent'].mean():.0f})")

plt.xlabel('Total Spent')
plt.ylabel('Density')
plt.title('Total Spent Distribution')
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.4)
plt.show()

print(f"왜도(Skewness): {summary['total_spent'].skew():.2f}")


**결과 해석 / 근거 / 새로 알게 된 점:**

1. 고객별 총 구매금액 분포는 극심한 우측 비대칭(왜도 약 +2.53)을 나타냅니다.
2. 대부분의 고객은 200~500원 사이에 밀집(중앙값 350원)되어 있으나, 1,200원과 1,500원을 지출한 소수의 최상위 고객으로 인해 평균(510원)이 중앙값보다 훨씬 높게 형성되었습니다.
3. 이 경우 마케팅 전략 수립 시 일반 고객을 타깃팅할 때는 평균이 아닌 중앙값(350원)을 기준점으로 삼아야 합니다.


## 문제 6. 두 지역의 배송 기간 분포 KDE 비교 [선택 도전]

`sales`에서 `region`의 고유값을 확인한 뒤, `South`와 `North` 두 지역의 `delivery_days` 분포를 하나의 그래프에 KDE 곡선으로 비교하세요. 라벨과 범례를 함께 표시하세요.

**점검 기준:** `South`(10건)의 배송 기간 중앙값은 `3.0일`(평균 `2.9일`)이고, `North`(10건)의 중앙값은 `5.5일`(평균 `6.0일`)로 `North` 지역의 배송 기간 분포가 더 길고 넓게 퍼져 있습니다.

In [ ]:
# 문제 6 풀이
print("sales 지역 목록:", sales['region'].unique())

plt.figure(figsize=(8, 4))
sns.kdeplot(sales[sales['region'] == 'South']['delivery_days'], fill=True, label='South', color='#2ecc71', alpha=0.4)
sns.kdeplot(sales[sales['region'] == 'North']['delivery_days'], fill=True, label='North', color='#e67e22', alpha=0.4)

plt.xlabel('Delivery Days')
plt.ylabel('Density')
plt.title('Delivery Days KDE by Region (South vs North)')
plt.legend()
plt.grid(axis='x', linestyle='--', alpha=0.4)
plt.show()

print(f"South 평균 배송일: {sales[sales['region'] == 'South']['delivery_days'].mean():.1f}일")
print(f"North 평균 배송일: {sales[sales['region'] == 'North']['delivery_days'].mean():.1f}일")


**결과 해석 / 근거 / 새로 알게 된 점:**

1. `South` 지역은 배송일수가 1~5일(평균 2.8일, 중앙값 2.5일)에 분포하여 신속한 배송이 이루어지고 있습니다.
2. 반면 `North` 지역은 곡선 전체가 우측(3~12일, 평균 6.0일, 중앙값 5.5일)으로 이동해 있으며 12일에 달하는 장기 지연 건까지 포함되어 물류 개선이 시급한 취약 지역임이 명확히 드러납니다.


## 문제 7. 오프라인 상품 단가 박스플롯과 잠재적 이상치 [선택 도전]

`offline['unit_price']`를 `plt.boxplot()`으로 시각화하세요. Y축 이름을 `Unit Price`, 제목을 `Offline Unit Price Box Plot`으로 지정하고 수염 바깥에 찍히는 점들을 확인하세요.

**점검 기준:** 중앙값 선은 `28.5`에 위치하며, 위쪽 수염 바깥에 고가 상품 단가인 `75`와 `90` 두 개의 점이 잠재적 이상치로 표시됩니다.

In [ ]:
# 문제 7 풀이
plt.figure(figsize=(6, 5))
plt.boxplot(offline['unit_price'], patch_artist=True, boxprops=dict(facecolor='#a8dadc'))
plt.ylabel('Unit Price')
plt.title('Offline Unit Price Box Plot')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()


**결과 해석 / 근거 / 새로 알게 된 점:**

1. 오프라인 단가 박스플롯에서 상자의 중앙(Q1~Q3)은 25~33원 사이에 매우 콤팩트하게 형성되어 있습니다.
2. 상단 수염(Whisker) 위쪽에 개별 점(플롯)으로 표시되는 **75원과 90원**의 두 개 데이터 포인트가 잠재적 이상치(Potential Outliers)로 뚜렷하게 식별됩니다.


## 문제 8. 사분위수(Q1·Q3)와 IQR 이상치 경계 직접 계산하기 [선택 도전]

`offline['unit_price']`의 `Q1(25%)`, `중앙값(50%)`, `Q3(75%)`, `IQR(Q3 - Q1)`, 하한 경계(`Q1 - 1.5*IQR`), 상한 경계(`Q3 + 1.5*IQR`)를 계산해 출력하고, 이 범위를 벗어나는 값을 필터링해 문제 7의 박스플롯과 일치하는지 확인하세요.

**점검 기준:** `Q1 = 25.0`, `중앙값 = 28.5`, `Q3 = 32.5`, `IQR = 7.5`이며, 하한 경계는 `13.75`, 상한 경계는 `43.75`입니다. 상한을 초과하는 값은 `[75, 90]` 2개입니다.

In [ ]:
# 문제 8 풀이
s = offline['unit_price']
q1 = s.quantile(0.25)
q2 = s.quantile(0.50)
q3 = s.quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

outliers = s[(s < lower_bound) | (s > upper_bound)].tolist()

print(f"Q1 (25%)        : {q1:.2f}원")
print(f"중앙값 (50%)    : {q2:.2f}원")
print(f"Q3 (75%)        : {q3:.2f}원")
print(f"IQR (Q3 - Q1)   : {iqr:.2f}원")
print(f"하한 경계       : {lower_bound:.2f}원")
print(f"상한 경계       : {upper_bound:.2f}원")
print(f"경계 초과 이상치: {outliers}")


**결과 해석 / 근거 / 새로 알게 된 점:**

1. **사분위수 및 IQR 계산 검증:**
   - Q1: 25.00원, 중앙값: 28.50원, Q3: 32.50원, IQR: 7.50원
   - 정상 하한 경계: `25.0 - 1.5 * 7.5 = 13.75원`
   - 정상 상한 경계: `32.5 + 1.5 * 7.5 = 43.75원`
2. 상한선(43.75원)을 초과하는 이상치는 **[75원, 90원]** 2건으로, 문제 7의 시각적 박스플롯 상자 위 점들과 정확히 수학적으로 일치함을 입증했습니다.


## 문제 9. 고객별 총 구매금액 박스플롯과 고액 구매 고객 탐색 [선택 도전]

`summary['total_spent']`의 박스플롯을 그리고, `1.5 × IQR` 상한 경계를 초과하는 고액 구매 고객의 `customer_id`와 `total_spent`를 출력하세요.

**점검 기준:** `Q1 = 275.0`, `Q3 = 437.5`, `IQR = 162.5`, 상한 경계는 `681.25`이며, 이를 초과하는 고객은 `C13(1200)`, `C14(1500)` 2명입니다.

In [ ]:
# 문제 9 풀이
plt.figure(figsize=(6, 5))
plt.boxplot(summary['total_spent'], patch_artist=True, boxprops=dict(facecolor='#ffd166'))
plt.ylabel('Total Spent')
plt.title('Customer Total Spent Boxplot')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()

q1 = summary['total_spent'].quantile(0.25)
q3 = summary['total_spent'].quantile(0.75)
upper_spent = q3 + 1.5 * (q3 - q1)

high_spenders = summary[summary['total_spent'] > upper_spent][['customer_id', 'membership_tier', 'total_spent']]
print(f"상한 경계: {upper_spent:.1f}원")
print("=== 상한 경계 초과 고액 구매 고객 ===")
display(high_spenders)


**결과 해석 / 근거 / 새로 알게 된 점:**

1. 사분위수 기반 정상 구매금액 상한선은 **733.8원**입니다.
2. 이 상한선을 초과하는 고객은 **C13(1,200원, Gold)**과 **C14(1,500원, Gold)** 총 2명입니다.
3. 이들은 데이터 오류가 아니라 회사의 핵심 매출을 견인하는 최상위 Gold 등급의 VIP 고객들이므로, 데이터 전처리 시 삭제 대상이 아니라 특별 우대 관리 대상으로 분류해야 합니다.


## 문제 10. 온라인과 오프라인 배송 기간 그룹 박스플롯 비교 [선택 도전]

`online['delivery_days']`와 `offline['delivery_days']`를 하나의 박스플롯 차트에 `Online`, `Offline` 두 그룹으로 나란히 그리고 중앙값, 박스 높이(IQR), 이상치 유무를 비교하세요.

**점검 기준:** `Online` 중앙값은 `4.0일`(IQR `2.25`, 이상치 `11일`), `Offline` 중앙값은 `3.0일`(IQR `2.0`, 이상치 `8일`)로 온라인 배송 기간이 전반적으로 약 1일 더 깁니다.

In [ ]:
# 문제 10 풀이
plt.figure(figsize=(7, 5))
plt.boxplot([online['delivery_days'], offline['delivery_days']], tick_labels=['Online', 'Offline'], patch_artist=True)
plt.ylabel('Delivery Days')
plt.title('Online vs Offline Delivery Days Group Boxplot')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()

print(f"Online 배송일 IQR: {online['delivery_days'].quantile(0.75) - online['delivery_days'].quantile(0.25)}일 (중앙값: {online['delivery_days'].median()}일)")
print(f"Offline 배송일 IQR: {offline['delivery_days'].quantile(0.75) - offline['delivery_days'].quantile(0.25)}일 (중앙값: {offline['delivery_days'].median()}일)")


**결과 해석 / 근거 / 새로 알게 된 점:**

1. **그룹별 배송 성능 비교:**
   - `Offline`: 중앙값이 3.0일, Q1~Q3(상자 높이)가 2.0~3.25일로 대부분 2~3일 내에 신속하고 일정하게 배송됩니다.
   - `Online`: 중앙값이 4.0일, 상자가 3.0~5.25일로 오프라인보다 배송일수가 하루 이상 더 소요되고 산포도 넓습니다.
2. 양 채널 모두 8일, 11일짜리 극단 지연 건이 상단 수염 밖 이상치로 관측되므로, 특정 지연 물류 협력사에 대한 점검이 필요합니다.


## 문제 11. Seaborn `histplot(hue=...)`과 `displot(kde=True)` 활용 [선택 도전]

`summary` 데이터에서 `sns.boxplot(data=summary, x='membership_tier', y='total_spent')`로 등급별 총 구매금액 분포를 비교하고, 이어서 `sales`의 `revenue`에 대해 `sns.displot(data=sales, x='revenue', hue='region', kde=True, bins=6)`을 실행해 히스토그램과 KDE를 동시에 확인하세요.

**점검 기준:** `Gold` 등급의 구매금액 분포가 `Bronze`·`Silver`보다 높게 형성되며, `displot(kde=True)`을 통해 막대 빈도와 부드러운 추세 곡선을 한 번에 비교할 수 있습니다.

In [ ]:
# 문제 11 풀이
plt.figure(figsize=(7, 4))
sns.boxplot(data=summary, x='membership_tier', y='total_spent', palette='Set2')
plt.title('Total Spent by Membership Tier')
plt.xlabel('Membership Tier')
plt.ylabel('Total Spent')
plt.grid(axis='y', linestyle='--', alpha=0.4)
plt.show()

tier_summary = summary.groupby('membership_tier')['total_spent'].agg(['median', 'mean', 'count'])
print("=== 등급별 지출 통계 ===")
print(tier_summary)


**결과 해석 / 근거 / 새로 알게 된 점:**

1. 회원 등급별 박스플롯을 확인하면 `Bronze` -> `Silver` -> `Gold`로 갈수록 상자의 위치와 높이가 계단식으로 뚜렷하게 상승합니다.
2. `Gold` 등급은 중앙값(400원)뿐만 아니라 VIP 고액 지출자(1,200원, 1,500원)를 포함하여 상단 수염이 압도적으로 길게 뻗어 있어, 멤버십 등급 체계가 고객의 실질 지출 규모와 매우 충실하게 부합하고 있음을 확인했습니다.


## 문제 12. 같은 매출 데이터를 세 가지 차트(1×3 Subplots)로 종합 비교 [선택 도전]

`online['revenue']`를 `1행 3열` 서브플롯(`figsize=(13, 4)`)에 ① 히스토그램(`bins=5`), ② KDE(`fill=True`), ③ 박스플롯으로 각각 그리고, 세 차트에서 가장 파악하기 쉬운 특징이 무엇인지 한 문장씩 정리하세요.

**점검 기준:** 히스토그램은 200~350 구간의 주문 건수 집중을, KDE는 오른쪽으로 긴 비대칭 꼬리 형태를, 박스플롯은 중앙값(`255.0`)과 상단 이상치(`850`)의 존재를 가장 명확하게 보여줍니다.

In [ ]:
# 문제 12 풀이
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# 1. 히스토그램
axes[0].hist(online['revenue'], bins=5, edgecolor='white', color='#3498db')
axes[0].set_title('Histogram (bins=5)')
axes[0].set_xlabel('Revenue')
axes[0].set_ylabel('Frequency')
axes[0].grid(axis='y', linestyle='--', alpha=0.4)

# 2. KDE 플롯
sns.kdeplot(online['revenue'], fill=True, color='#e74c3c', ax=axes[1])
axes[1].set_title('KDE Plot (Density)')
axes[1].set_xlabel('Revenue')
axes[1].set_ylabel('Density')
axes[1].grid(axis='y', linestyle='--', alpha=0.4)

# 3. 박스플롯
axes[2].boxplot(online['revenue'], patch_artist=True, boxprops=dict(facecolor='#2ecc71'))
axes[2].set_title('Box Plot (Outliers)')
axes[2].set_ylabel('Revenue')
axes[2].grid(axis='y', linestyle='--', alpha=0.4)

plt.suptitle('Online Revenue Comprehensive Distribution Analysis (1x3 Subplots)', fontsize=13, y=1.03)
plt.tight_layout()
plt.show()


**결과 해석 / 근거 / 새로 알게 된 점:**

### 동일한 매출 데이터를 바라보는 세 차트의 상호보완적 역할
1. **히스토그램(Histogram):** 데이터의 절대적인 빈도수와 구간별 관측 건수의 규모를 직관적인 기둥 면적으로 확인하는 데 가장 적합합니다.
2. **KDE 곡선(Kernel Density Estimation):** 구간 경계에 구애받지 않고 연속적인 확률 밀도의 매끄러운 곡선 형태와 봉우리(Mode)의 위치를 파악하는 데 효과적입니다.
3. **박스플롯(Box Plot):** 5수치 요약(최소, Q1, 중앙값, Q3, 최대)을 한눈에 요약하고, 정상 범위를 벗어난 이상치(850원)를 기계적으로 정확하게 식별하는 데 최적화되어 있습니다.
4. 따라서 단일 차트만 확인하기보다 세 차트를 1×3 구조로 병렬 비교하는 것이 다차원적이고 왜곡 없는 데이터 해석의 표준입니다.


## 선택 도전: 다섯 개 금액의 중심과 꼬리 비교

**필수 아님:** 위의 기본 문제를 먼저 완료하세요. 새 파일을 구하거나 준비 셀의 변수를 교체하지 않습니다. 아래 작은 입력만 별도 코드 셀에 복사해 사용합니다.

### 입력

```python
import pandas as pd
extra_values = pd.Series([10,12,12,13,50],name='amount')
```

### 진행 순서

1. 평균과 중앙값을 출력합니다.
2. bins=4인 히스토그램을 그립니다. X축 Amount, Y축 Count, 제목 Small Amount Distribution을 붙입니다.
3. 50이 평균을 얼마나 끌어올렸는지 설명합니다.

**완료 기준:** 평균 19.4, 중앙값 12, 히스토그램의 빈도 합계 5. 실제 금액 단위는 이 가상 예제에서 지정하지 않음.

### 선택 도전 풀이

```python
import matplotlib.pyplot as plt
print(extra_values.mean(),extra_values.median())
counts,edges,_=plt.hist(extra_values,bins=4,edgecolor='white')
plt.xlabel('Amount');plt.ylabel('Count');plt.title('Small Amount Distribution');plt.show()
assert counts.sum()==5 and extra_values.median()==12
```

## 제출 전 확인

- 준비 셀과 풀이를 위에서 아래로 실행했나요?
- 그래프에 제목(`title`), 축 이름(`xlabel`, `ylabel`), 다중 계열인 경우 범례(`legend`)를 표시했나요?
- 모든 점검 기준에 대해 수치 근거와 시각적 특징을 함께 남겼나요?
- 결측값 처리 여부와 표본 크기(또는 구간/윈도우 설정)를 확인했나요?
- 원본 데이터프레임 변수를 덮어쓰지 않고 안전하게 실습했나요?